In [ ]:
%pip install ChemLogic

In [ ]:
from chemlogic.utils.Pipeline import Pipeline

# Small dummy dataset — replace with your own SMILES and labels
smiles = [
    "CCO",
    "CC(=O)O",
    "c1ccccc1",
    "CC(C)O",
    "CCCO",
    "c1ccc(O)cc1",
    "CC(=O)c1ccccc1",
    "CCN",
    "c1ccncc1",
    "CC(=O)Nc1ccccc1",
    "OC(=O)c1ccccc1",
    "Cc1ccccc1",
    "CCOCC",
    "CC(C)=O",
    "OCC",
    "CC(N)=O",
    "c1ccc(N)cc1",
    "CC(O)C",
    "CCC(=O)O",
    "c1ccc(Cl)cc1",
]
target = [
    4.2,
    5.1,
    3.8,
    4.6,
    4.0,
    5.5,
    5.8,
    3.5,
    4.9,
    6.1,
    5.3,
    4.1,
    3.7,
    4.4,
    3.9,
    5.0,
    5.2,
    4.3,
    4.7,
    5.6,
]
logp = [
    -0.18,
    -0.17,
    2.13,
    0.05,
    0.25,
    1.46,
    1.58,
    -0.13,
    0.65,
    1.16,
    1.87,
    2.61,
    1.05,
    -0.24,
    -0.92,
    -0.39,
    0.90,
    0.05,
    0.33,
    2.86,
]

# Train a regression model
pipeline = Pipeline(
    dataset_name="checkpoint_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=smiles,
    labels=target,
)

train_loss, test_loss, r2, _ = pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, r2)

In [ ]:
# Save the trained model to a checkpoint
safetensors_path, json_path = pipeline.save_checkpoint(
    "../checkpoints/checkpoint_demo",
    metadata={"description": "regression demo"},
)
print(f"Saved: {safetensors_path}")
print(f"       {json_path}")

In [ ]:
# Reload the checkpoint and run inference on new molecules
loaded = Pipeline.from_checkpoint(
    "../checkpoints/checkpoint_demo",
    smiles_list=smiles,
    labels=target,
)

new_smiles = ["CCCC", "c1ccc(F)cc1", "CC(=O)OCC"]
predictions = loaded.inference(new_smiles)

for smi, pred in zip(new_smiles, predictions):
    print(f"{smi:25s}  target = {pred:.4f}")

In [ ]:
# Multi-class: 3 activity classes — pass task= explicitly (integers are ambiguous)
import pandas as pd

mc_labels = pd.cut(target, bins=3, labels=[0, 1, 2]).astype(int).tolist()

mc_pipeline = Pipeline(
    dataset_name="multiclass_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=smiles,
    labels=mc_labels,
    task="multi_class",
    num_outputs=3,
)

train_loss, test_loss, auroc_ovr, _ = mc_pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, auroc_ovr)

In [ ]:
# Multi-regression: predict target and logP simultaneously
mr_labels = list(zip(target, logp))

mr_pipeline = Pipeline(
    dataset_name="multiregression_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=smiles,
    labels=mr_labels,
    task="multi_regression",
    num_outputs=2,
)

train_loss, test_loss, r2, _ = mr_pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, r2)